# Find Synopses


# Setup


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import json

# Make Flat

In [11]:
directory_path = "..\\..\\raw"
all_parsed_files = glob.glob(os.path.join(directory_path, "*.csv"))
# augmentation_files = [filename for filename in all_parsed_files if "20" in filename]
all_parsed_files

['..\\..\\raw\\0_keep_first.csv',
 '..\\..\\raw\\0_r2_dupe_keep_first.csv',
 '..\\..\\raw\\2021_Jan_Dec.csv',
 '..\\..\\raw\\2022_Jan_June.csv',
 '..\\..\\raw\\2022_July_Dec.csv',
 '..\\..\\raw\\2023_Jan_Dec.csv',
 '..\\..\\raw\\2024_Jan_June.csv',
 '..\\..\\raw\\2024_July_Dec.csv',
 '..\\..\\raw\\2025_Jan_June.csv',
 '..\\..\\raw\\2025_July_Dec.csv',
 '..\\..\\raw\\airspace_structure_jan_2015_jan_2021.csv',
 '..\\..\\raw\\atc_nav_builds_jan_2015_jan_2021.csv',
 '..\\..\\raw\\chartorpub_jan_2015_jan_2021.csv',
 '..\\..\\raw\\company_policy_jan_2015_jan_2021.csv',
 '..\\..\\raw\\env_nonweather_jan_2015_jan_2021.csv',
 '..\\..\\raw\\equipment_tooling_jan_2015_jan_2021.csv',
 '..\\..\\raw\\humanfactors_jan_2020_jan_2021.csv',
 '..\\..\\raw\\incorrect_part_jan_2015_jan_2021.csv',
 '..\\..\\raw\\logbook_entry_jan_2015_jan_2021.csv',
 '..\\..\\raw\\manuals_jan_2015_jan_2021.csv',
 '..\\..\\raw\\mel_jan_2015_jan_2021.csv',
 '..\\..\\raw\\merged_2015_2025.csv',
 '..\\..\\raw\\merged_2021_2025.

In [ ]:
li = []

# Read all CSVs
for filename in all_parsed_files:
    if "merged" in filename:
        continue
    df = pd.read_csv(filename, index_col=0, header=[0, 1], low_memory=False)
    df = df.iloc[:, :-1]
    li.append(df)

# Combine all DataFrames
df = pd.concat(li, axis=0)

# Remove 'unnamed' column at end (because the CSVs contain ',\n' by default)

# Remove those with duplicate ACNs
unique = df[~df.index.duplicated(keep="first")]

In [27]:
flat = unique.copy()
flat.columns = [f"{a.replace(' ', "_")}_{b.replace(' ', "_")}" for a, b in flat.columns]
flat.index.name = "acn"
flat.head()

,Time_Date,Time_Local_Time_Of_Day,Place_Locale_Reference,Place_State_Reference,Place_Relative_Position.Angle.Radial,Place_Relative_Position.Distance.Nautical_Miles,Place_Altitude.AGL.Single_Value,Place_Altitude.MSL.Single_Value,Place_Latitude_/_Longitude_(UAS),Environment_Flight_Conditions,...,Events_Detector,Events_When_Detected,Events_Result,Assessments_Contributing_Factors_/_Situations,Assessments_Primary_Problem,Report_1_Narrative,Report_1_Callback,Report_2_Narrative,Report_2_Callback,Report_1_Synopsis
acn,,,,,,,,,,,,,,,,,,,,,
2260174,202507,1201-1800,BWI.Airport,MD,NaN,NaN,NaN,900.0,NaN,VMC,...,Person Air Traffic Control,In-flight,General None Reported / Taken,Procedure; Human Factors; ATC Equipment / Nav ...,Ambiguous,Aircraft vectored in within 1NM to final appro...,NaN,NaN,NaN,Air carrier Captain reported a low altitude al...
2260249,202507,0601-1200,ZZZ.Airport,US,NaN,NaN,NaN,NaN,NaN,NaN,...,Automation Aircraft Other Automation; Person F...,In-flight,Flight Crew Executed Go Around / Missed Approa...,Human Factors; Aircraft,Ambiguous,While on short final we received a glideslope ...,NaN,NaN,NaN,Air carrier pilot reported receiving an aircra...
2260370,202507,1801-2400,ZZZ.ARTCC,US,NaN,NaN,NaN,35000.0,NaN,VMC,...,Person Flight Crew,In-flight,Air Traffic Control Issued New Clearance; Air ...,Aircraft,Aircraft,Flying at cruise; FL350; the FO was the PF and...,NaN,At cruise; FL350; during level-off; the Captai...,NaN,B737 flight crew reported observing a slowing ...
2261277,202507,1801-2400,ZZZ.Airport,US,NaN,2.6,160.0,NaN,NaN,NaN,...,Person Flight Crew,In-flight,General None Reported / Taken,Human Factors,Human Factors,On Day 0 around XA:30; I forgot to get LAANC a...,Reporter stated TRUST certificate was obtained...,NaN,NaN,Recreational / Hobbyist UAS pilot reported fly...
2261317,202507,NaN,ZZZ.Airport,US,NaN,NaN,NaN,9000.0,NaN,VMC,...,Automation Aircraft Other Automation; Person F...,In-flight,Flight Crew Became Reoriented; Flight Crew Reg...,Weather; Human Factors; Procedure,Procedure,Divert into ZZZ from ZZZ1. FO flying. Vectors ...,NaN,Extremely strong winds blew us off the LOC whi...,NaN,B737 flight crew reported the pilot flying in ...


In [28]:
flat.to_csv("flattened.csv")

# Make Training Synopses

In [ ]:
import datasets
train_ds = datasets.load_dataset("sookiemonster/asrs-narratives-rebalance", split="train")

c:\Users\legos\Desktop\Work\Github\asrs-problem-factor-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\legos\Desktop\Work\Github\asrs-problem-factor-analysis\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\legos\.cache\huggingface\hub\datasets--sookiemonster--asrs-narratives-rebalance. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to

In [9]:
train_df = train_ds.to_pandas()
train_acns = train_df['acn'].astype('int').to_list()
train_acns[:5]

[2008766, 2109432, 1974739, 1904026, 1787366]

In [17]:
flat = pd.read_csv("../flattened.csv", index_col=0, low_memory=False)
syn = flat[['Report_1_Synopsis', 'Assessments_Primary_Problem']]
syn.index = syn.index.astype('int')

display(syn.head())

,Report_1_Synopsis,Assessments_Primary_Problem
acn,,
2260174,Air carrier Captain reported a low altitude al...,Ambiguous
2260249,Air carrier pilot reported receiving an aircra...,Ambiguous
2260370,B737 flight crew reported observing a slowing ...,Aircraft
2261277,Recreational / Hobbyist UAS pilot reported fly...,Human Factors
2261317,B737 flight crew reported the pilot flying in ...,Procedure


In [ ]:
train_synopses = syn.loc[train_acns]
display(train_synopses)
train_synopses.to_csv("train_synopses.csv")

,Report_1_Synopsis,Assessments_Primary_Problem
acn,,
2008766,A TRACON Controller reported aircraft on appro...,Weather
2109432,Air carrier Captain reported the Ramp Agent ga...,Human Factors
1974739,BE35 pilot reported loss of oil pressure in fl...,Human Factors
1904026,B767 Captain reported a gate return due to exc...,Aircraft
1787366,C172 flight instructor reported a complete fai...,Aircraft
...,...,...
1269926,Ramp worker reported a 'significant weight shi...,Company Policy
1537257,ERJ-175 First Officer reported N90 TRACON requ...,Procedure
1481252,ZLA Center controller reported they received a...,Procedure


In [25]:
remove = set(
    [
        "Ambiguous",
        "Human Factors",
        "Procedure",
        "Aircraft",
        "Weather",
        "Human Factors; Human Factors",
        "Aircraft; Aircraft",
        "Procedure; Procedure",
        "Ambiguous; Ambiguous",
    ]
)
rest = train_synopses[~train_synopses["Assessments_Primary_Problem"].isin(remove)]
rest.to_csv("rest_train_synopses.csv")
rest

,Report_1_Synopsis,Assessments_Primary_Problem
acn,,
1897807,B737 Flight Crew reported a bird strike right ...,Environment - Non Weather Related
1785691,Air Taxi flight crew reported not hearing ATC ...,Environment - Non Weather Related
2096340,B737-800 flight crew reported encountering mul...,Environment - Non Weather Related
1781064,Air carrier flight crew reported a passenger m...,Environment - Non Weather Related
1901569,EMB-145 First Officer reported encountering wa...,Environment - Non Weather Related
...,...,...
1454345,An Approach Controller reported that a Local C...,Manuals
1276838,ZOB Controller reports of a Controller coming ...,Company Policy
1749386,Ground employee reported that co-workers are n...,Company Policy


In [26]:
rest.Assessments_Primary_Problem.value_counts()

Assessments_Primary_Problem
Company Policy                                  1281
Environment - Non Weather Related               1123
Chart Or Publication                             701
Airspace Structure                               664
ATC Equipment / Nav Facility / Buildings         602
Equipment / Tooling                              357
Airport                                          256
Staffing                                         226
Manuals                                          173
MEL                                              148
Incorrect / Not Installed / Unavailable Part     112
Software and Automation                           71
Logbook Entry                                     26
Name: count, dtype: int64